# Chapter 5: where the API key goes

This notebook runs the statements in `key-demos.cypher`, one per cell, so you can see each result as you go. The text above each cell is the comment from the script.

- Run the cells one at a time, from the top.
- Needs the 500 transactions loaded in chapter 2 and TYPESAFE_API_KEY set.

## 1. Install Dependencies

In [ ]:
%pip install neo4j==5.28.1 \
             pandas==2.2.3 --quiet

print("Install complete.")

## 2. Imports

In [ ]:
import json
import os
import pandas as pd

from IPython.display import display
from neo4j import GraphDatabase

## 3. Configuration

The connection details come from environment variables set before Jupyter was launched.

In [ ]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

## 4. Connect to Neo4j

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

print("Connected to", NEO4J_URI)

## 5. Helper

`run_cypher` runs one statement. A single map, such as a `jev.decide` result, prints as JSON. Anything else shows as a table.

In [ ]:
def run_cypher(query):
    """Run one statement and show the result. A single map prints as JSON, anything
    else as a table. Statements that change the graph also report what changed."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        records = [r.data() for r in result]
        summary = result.consume()

    if len(records) == 1 and len(records[0]) == 1 and isinstance(next(iter(records[0].values())), dict):
        print(json.dumps(next(iter(records[0].values())), indent=2, default=str))
    elif records:
        display(pd.DataFrame(records))
    else:
        print("No results.")

    changes = {name: getattr(summary.counters, name, 0) for name in (
        "nodes_created", "nodes_deleted", "relationships_created",
        "relationships_deleted", "properties_set", "constraints_added",
        "constraints_removed")}
    changes = {k: v for k, v in changes.items() if v}
    if changes:
        print("Changes:", ", ".join(f"{k} {v}" for k, v in changes.items()))
    ms = (summary.result_available_after or 0) + (summary.result_consumed_after or 0)
    print(f"Completed after {ms:,} ms")

## 6. The statements

**1.** A normal call to the real endpoint. The key is sent, so metadata.authenticated is true.

In [ ]:
run_cypher("""
MATCH (t:Transaction {txn_id: 'T0001'})
RETURN jev.decide(
  t {.amount, .merchant_category, .distance_from_home_km, .hour_of_day,
     .account_age_days, .prior_flags, .txns_last_24h, .typical_monthly_spend},
  {decision: {
     type: 'choice',
     instructions: 'Based on all available signals, should this transaction be flagged for review or passed through?',
     criteria: {
       Flag: 'Potential fraud. Send for review.',
       Pass: 'Likely legitimate. Process normally.'
     }
  }}
) AS result
""")

**2.** A different endpoint: nothing listens on port 9 of your own machine, so the connection is refused. The error names the host and port, one attempt is made and the key is not sent, so authenticated is false.

In [ ]:
run_cypher("""
RETURN jev.decide(
  {amount: 120.5, merchant_category: 'grocery'},
  {decision: {
     type: 'choice',
     instructions: 'Should this transaction be flagged or passed?',
     criteria: {Flag: 'Potential fraud.', Pass: 'Likely legitimate.'}
  }},
  {url: 'http://127.0.0.1:9/v1/systemone', max_retries: 0}
) AS result
""")

**3.** The same call with the default retries: three attempts with waits of 0.5 s and 1 s between them, so it takes at least 1.5 s.

In [ ]:
run_cypher("""
RETURN jev.decide(
  {amount: 120.5, merchant_category: 'grocery'},
  {decision: {
     type: 'choice',
     instructions: 'Should this transaction be flagged or passed?',
     criteria: {Flag: 'Potential fraud.', Pass: 'Likely legitimate.'}
  }},
  {url: 'http://127.0.0.1:9/v1/systemone'}
) AS result
""")

## 7. Close the Connection

In [ ]:
driver.close()

print("Connection closed.")